In [1]:
%load_ext autotime

time: 114 µs (started: 2026-09-28 22:13:38 +05:30)


In [2]:
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print(f"mlflow {mlflow.__version__} · scikit-learn {sklearn.__version__} · pandas {pd.__version__}")

mlflow 3.16.1 · scikit-learn 1.7.2 · pandas 2.3.3
time: 2.27 s (started: 2026-09-28 22:14:04 +05:30)


In [5]:
pd.options.display.max_columns = 100
pd.options.display.max_colwidth = None

time: 202 µs (started: 2026-09-28 22:14:57 +05:30)


In [6]:
bookings = pd.read_csv("data/bookings_2015_2017.csv.gz")

print(f"{len(bookings):,} bookings · arrivals {bookings.arrival_date.min()} → {bookings.arrival_date.max()}")
print(f"cancelled: {bookings.is_canceled.mean():.1%}")
bookings.head()

118,300 bookings · arrivals 2015-07-01 → 2017-08-24
cancelled: 37.1%


,booking_id,arrival_date,hotel,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,customer_type,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,is_canceled
0,BK100001,2015-07-01,City Hotel,6,0,2,1,0.0,0,HB,Offline TA/TO,TA/TO,0,0,0,A,No Deposit,Transient,0,0.0,0,0,0
1,BK100002,2015-07-01,City Hotel,88,0,4,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,Transient,0,76.5,0,1,1
2,BK100003,2015-07-01,City Hotel,65,0,4,1,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,Transient,0,68.0,0,1,1
3,BK100004,2015-07-01,City Hotel,92,2,4,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,Transient,0,76.5,0,2,1
4,BK100005,2015-07-01,City Hotel,257,0,2,2,0.0,0,HB,Offline TA/TO,TA/TO,0,1,0,A,No Deposit,Transient,0,101.5,0,0,1


time: 130 ms (started: 2026-09-28 22:15:00 +05:30)


In [7]:
NUMERIC = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children",
    "babies", "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
    "days_in_waiting_list", "adr", "required_car_parking_spaces", "total_of_special_requests",
]
CATEGORICAL = [
    "hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type",
    "deposit_type", "customer_type",
]
FEATURES = NUMERIC + CATEGORICAL
SEED = 42

X, y = bookings[FEATURES], bookings["is_canceled"]
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=SEED)

print(f"train {len(X_train):,} · validation {len(X_val):,} · test {len(X_test):,} · {len(FEATURES)} features")

train 70,980 · validation 23,660 · test 23,660 · 20 features
time: 47 ms (started: 2026-09-28 22:19:02 +05:30)


In [8]:
def build_pipeline(params):
    # params = {"model_type": ..., **hyperparameters}  →  an untrained preprocessing + model Pipeline
    hyper = {k: v for k, v in params.items() if k != "model_type"}
    if params["model_type"] == "logreg":
        model = LogisticRegression(max_iter=1000, **hyper)
    elif params["model_type"] == "random_forest":
        model = RandomForestClassifier(n_jobs=-1, random_state=SEED, **hyper)
    elif params["model_type"] == "hist_gb":
        model = HistGradientBoostingClassifier(random_state=SEED, **hyper)
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("prep", prep), ("model", model)])


def evaluate(pipe, X, y, prefix="val"):
    proba = pipe.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        f"{prefix}_roc_auc": roc_auc_score(y, proba),
        f"{prefix}_pr_auc": average_precision_score(y, proba),
        f"{prefix}_f1": f1_score(y, pred),
        f"{prefix}_precision": precision_score(y, pred),
        f"{prefix}_recall": recall_score(y, pred),
    }

time: 830 µs (started: 2026-09-28 22:22:01 +05:30)


In [9]:
for C in [0.1, 1.0]:
    pipe = build_pipeline({"model_type": "logreg", "C": C}).fit(X_train, y_train)
    print(f"logreg C={C}:  val ROC-AUC {evaluate(pipe, X_val, y_val)['val_roc_auc']:.4f}")

logreg C=0.1:  val ROC-AUC 0.8471
logreg C=1.0:  val ROC-AUC 0.8474
time: 640 ms (started: 2026-09-28 22:24:05 +05:30)


In [10]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

time: 4.45 ms (started: 2026-09-28 22:25:06 +05:30)


In [11]:
experiment = mlflow.set_experiment("booking-cancellations")

print("tracking URI     :", mlflow.get_tracking_uri())
print("experiment       :", experiment.name, "| id", experiment.experiment_id)
print("artifact location:", experiment.artifact_location)

2026/09/28 22:25:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/28 22:25:20 INFO mlflow.store.db.utils: Updating database tables
2026/09/28 22:25:21 INFO mlflow.tracking.fluent: Experiment with name 'booking-cancellations' does not exist. Creating a new experiment.


tracking URI     : sqlite:///mlflow.db
experiment       : booking-cancellations | id 1
artifact location: /Users/shivam13juna/Documents/scaler/AIML_mlops/v1-aiml-mlops/lec5 Experiment Tracking with MLFlow/mlruns/1
time: 820 ms (started: 2026-09-28 22:25:20 +05:30)


In [12]:
params = {"model_type": "logreg", "C": 1.0}

with mlflow.start_run(
    run_name="baseline-logreg",
    description="Logistic regression on the 20 raw features: the bar every other model has to clear.",
) as run:
    mlflow.log_params(params)
    mlflow.set_tags({"stage": "baseline", "team": "revenue-analytics"})

    start = time.time()
    pipe = build_pipeline(params).fit(X_train, y_train)
    mlflow.log_metric("fit_time_s", time.time() - start)

    metrics = evaluate(pipe, X_val, y_val)
    mlflow.log_metrics(metrics)

print("run_id:", run.info.run_id)
pd.Series(metrics).round(4)

run_id: 5603fd197f1a42c78006fca20d371b75


val_roc_auc      0.8474
val_pr_auc       0.8226
val_f1           0.6965
val_precision    0.8449
val_recall       0.5924
dtype: float64

time: 468 ms (started: 2026-09-28 22:27:14 +05:30)


In [13]:
logged = mlflow.get_run(run.info.run_id)

print("status :", logged.info.status)
print("params :", logged.data.params)
print("metrics:", {k: round(v, 4) for k, v in logged.data.metrics.items()})
print("tags   :")
for key, value in sorted(logged.data.tags.items()):
    print(f"   {key:<22} {value}")

status : FINISHED
params : {'model_type': 'logreg', 'C': '1.0'}
metrics: {'fit_time_s': 0.3244, 'val_roc_auc': 0.8474, 'val_pr_auc': 0.8226, 'val_f1': 0.6965, 'val_precision': 0.8449, 'val_recall': 0.5924}
tags   :
   mlflow.note.content    Logistic regression on the 20 raw features: the bar every other model has to clear.
   mlflow.runName         baseline-logreg
   mlflow.source.name     Live_Class_Notebook.ipynb
   mlflow.source.type     NOTEBOOK
   mlflow.user            shivam13juna
   stage                  baseline
   team                   revenue-analytics
time: 6.32 ms (started: 2026-09-28 22:32:46 +05:30)


## Let's Sweep

In [14]:
candidates = {
    "logreg-C0.1":       {"model_type": "logreg", "C": 0.1},
    "rf-200-d12":        {"model_type": "random_forest", "n_estimators": 200, "max_depth": 12},
    "rf-200-d20":        {"model_type": "random_forest", "n_estimators": 200, "max_depth": 20},
    "rf-200-d30":        {"model_type": "random_forest", "n_estimators": 200, "max_depth": 30},
    "rf-400-d30":        {"model_type": "random_forest", "n_estimators": 400, "max_depth": 30},
    "hist_gb-31leaves":  {"model_type": "hist_gb", "learning_rate": 0.1, "max_leaf_nodes": 31, "max_iter": 300},
    "hist_gb-127leaves": {"model_type": "hist_gb", "learning_rate": 0.1, "max_leaf_nodes": 127, "max_iter": 300},
}

for run_name, params in candidates.items():
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        mlflow.log_dict(params, "config.json")
        mlflow.set_tag("stage", "sweep")

        start = time.time()
        pipe = build_pipeline(params).fit(X_train, y_train)
        fit_time = time.time() - start
        size_mb = len(pickle.dumps(pipe)) / 1e6
        metrics = evaluate(pipe, X_val, y_val)
        mlflow.log_metrics({**metrics, "fit_time_s": fit_time, "model_size_mb": size_mb})

        if params["model_type"] == "hist_gb":  # one loss value per boosting round → a curve
            booster = pipe.named_steps["model"]
            for step, (train_score, holdout_score) in enumerate(
                zip(booster.train_score_, booster.validation_score_)
            ):
                mlflow.log_metrics({"train_loss": -train_score, "holdout_loss": -holdout_score}, step=step)

    print(f"{run_name:<18} val_roc_auc {metrics['val_roc_auc']:.4f}   fit {fit_time:5.1f}s   {size_mb:7.1f} MB")

logreg-C0.1        val_roc_auc 0.8471   fit   0.3s       0.0 MB
rf-200-d12         val_roc_auc 0.8889   fit   0.7s      15.8 MB
rf-200-d20         val_roc_auc 0.9099   fit   0.8s     109.7 MB
rf-200-d30         val_roc_auc 0.9159   fit   0.9s     310.4 MB
rf-400-d30         val_roc_auc 0.9163   fit   1.8s     617.0 MB
hist_gb-31leaves   val_roc_auc 0.9040   fit   6.8s       1.1 MB
hist_gb-127leaves  val_roc_auc 0.9093   fit  11.4s       2.3 MB
time: 24.5 s (started: 2026-09-28 22:37:04 +05:30)


In [15]:
sweep = mlflow.search_runs(
    experiment_names=["booking-cancellations"],
    filter_string="tags.stage = 'sweep' and attributes.status = 'FINISHED'",
    order_by=["metrics.val_roc_auc DESC"],
)

sweep[["tags.mlflow.runName", "metrics.val_roc_auc", "metrics.val_pr_auc", "metrics.val_f1",
       "metrics.model_size_mb", "metrics.fit_time_s"]].round(3)

,tags.mlflow.runName,metrics.val_roc_auc,metrics.val_pr_auc,metrics.val_f1,metrics.model_size_mb,metrics.fit_time_s
0,rf-400-d30,0.916,0.895,0.786,617.019,1.786
1,rf-200-d30,0.916,0.895,0.787,310.422,0.901
2,rf-200-d20,0.910,0.884,0.762,109.696,0.771
3,hist_gb-127leaves,0.909,0.884,0.765,2.264,11.365
4,hist_gb-31leaves,0.904,0.877,0.753,1.087,6.762
5,rf-200-d12,0.889,0.859,0.714,15.833,0.652
6,logreg-C0.1,0.847,0.822,0.696,0.005,0.258


time: 23.7 ms (started: 2026-09-28 22:38:26 +05:30)
